In [1]:
from pathlib import Path
import pandas as pd
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.io import load_parquet
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 100)

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet

amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

import re

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', '', t)
    # only change: space out dots between letters so initials stay separate
    t = re.sub(r'(?<=\b[a-z])\.\s*(?=[a-z]\b)', ' ', t)
    t = re.sub(r'[^a-z0-9\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def clean_author_field_goodreads(author_str: str) -> str:
    if not isinstance(author_str, str):
        return ""
    authors = []
    parts = [p.strip() for p in author_str.split(",") if p.strip()]
    for i, raw in enumerate(parts):
        has_paren = "(" in raw
        name = re.split(r"\s*\(", raw)[0].strip()  # drop parenthetical
        if len(name.split()) < 2:  # need first + surname
            continue
        if i == 0:
            authors.append(name)
            continue
        if has_paren:
            break  # stop at first later author with parentheses
        authors.append(name)
    return ", ".join(authors)


amazon_sample['clean_title'] = amazon_sample['title'].apply(clean_text)
goodreads_sample['clean_title'] = goodreads_sample['title'].apply(clean_text)
metabooks_sample['clean_title'] = metabooks_sample['title'].apply(clean_text)
amazon_sample["clean_author"] = amazon_sample["author"].apply(clean_text)
# We have an extra step for goodreads authors since it contains unnecessary author info in the author field
goodreads_sample["clean_author"] = goodreads_sample["author"].apply(clean_author_field_goodreads)
goodreads_sample["clean_author"] = goodreads_sample["clean_author"].apply(clean_text)
metabooks_sample["clean_author"] = metabooks_sample["author"].apply(clean_text)
amazon_sample["clean_publisher"] = amazon_sample["publisher"].apply(clean_text)
metabooks_sample["clean_publisher"] = metabooks_sample["publisher"].apply(clean_text)
goodreads_sample["clean_publisher"] = goodreads_sample["publisher"].apply(clean_text)

In [3]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [4]:
from PyDI.entitymatching import EmbeddingBlocker

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks_sample, amazon_sample,
    text_cols=['clean_title','clean_author','clean_publisher','publish_year'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=2,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks_sample, goodreads_sample,
    text_cols=['clean_title','clean_author','clean_publisher','publish_year'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=2,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

embedding_candidates_m2a = embedding_blocker_m2a.materialize()
embedding_candidates_m2g = embedding_blocker_m2g.materialize()

In [5]:
from PyDI.entitymatching import EntityMatchingEvaluator

# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2a,
    blocker=embedding_blocker_m2a,
    test_pairs=train_m2a,
    out_dir=BLOCK_EVAL_DIR
)
# evaluate blocking: metabooks -> goodreads
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=embedding_candidates_m2g,
    blocker=embedding_blocker_m2g,
    test_pairs=train_m2g,
    out_dir=BLOCK_EVAL_DIR
)

display(results_m2a)
display(results_m2g)

{'pair_completeness': 0.9652567975830816,
 'pair_quality': 0.009158664182313316,
 'reduction_ratio': 0.9999428587808453,
 'total_candidates': 69770,
 'total_possible_pairs': 1221010000,
 'true_positives_found': 639,
 'total_true_pairs': 662,
 'evaluation_timestamp': '2025-12-20T02:42:15.847611',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

{'pair_completeness': 0.96,
 'pair_quality': 0.007911596508578063,
 'reduction_ratio': 0.9999428579618512,
 'total_candidates': 69771,
 'total_possible_pairs': 1221010000,
 'true_positives_found': 552,
 'total_true_pairs': 575,
 'evaluation_timestamp': '2025-12-20T02:42:18.198147',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [38]:
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import StringComparator, NumericComparator

# Base comparators used by both combos
comparators = [
    StringComparator(column="clean_title", similarity_function="cosine"),
    StringComparator(column="clean_author", similarity_function="cosine"),
    NumericComparator(column="publish_year", method="within_range", max_difference=0),
    StringComparator(column="clean_publisher", similarity_function="cosine"),
]

matcher = RuleBasedMatcher()

# matching metabooks > amazon
correspondences_m2a = matcher.match(
    df_left=metabooks_sample,
    df_right=amazon_sample, 
    candidates=embedding_blocker_m2a,
    comparators=comparators,
    weights=[0.5, 0.3, 0.1,0.1], 
    threshold=0.7,
    id_column='id'
)

# matching metabooks > goodreads
correspondences_m2g = matcher.match(
    df_left=metabooks_sample,
    df_right=goodreads_sample, 
    candidates=embedding_blocker_m2g,
    comparators=comparators,
    weights=[0.5, 0.3, 0.1,0.1], 
    threshold=0.7,
    id_column='id'
)

In [39]:
debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

# evaluate matching metabooks -> amazon
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2a,
    test_pairs=test_m2a
)

display(eval_results_m2a)

{'precision': 0.9539473684210527,
 'recall': 0.8734939759036144,
 'f1': 0.9119496855345911,
 'accuracy': 0.8847736625514403,
 'true_positives': 145,
 'false_positives': 7,
 'false_negatives': 21,
 'true_negatives': 70,
 'threshold_used': 0.0,
 'total_correspondences': 24816,
 'filtered_correspondences': 24816,
 'evaluation_timestamp': '2025-12-20T03:04:40.863106'}

In [40]:
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 0.9069767441860465,
 'recall': 0.8125,
 'f1': 0.8571428571428572,
 'accuracy': 0.8368200836820083,
 'true_positives': 117,
 'false_positives': 12,
 'false_negatives': 27,
 'true_negatives': 83,
 'threshold_used': 0.0,
 'total_correspondences': 6299,
 'filtered_correspondences': 6299,
 'evaluation_timestamp': '2025-12-20T03:04:46.406002',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [41]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,20045,92.908459
1,3,607,2.813441
2,4,801,3.712630
3,5,63,0.292005
4,6,42,0.194670
5,7,8,0.037080
6,8,6,0.027810
7,9,2,0.009270
8,10,1,0.004635



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5094,90.431386
1,3,459,8.148411
2,4,57,1.011894
3,5,13,0.230783
4,6,7,0.124268
5,7,3,0.053258


In [ ]:
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2a,
    out_path=cluster_analysis_dir/"m2a_clusters_rb.json")
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2g,
    out_path=cluster_analysis_dir/"m2g_clusters_rb.json")


PosixPath('/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_rb.json')

In [6]:
import json
import pandas as pd
from pathlib import Path

# load the clusters file
clusters = json.loads(Path(ROOT/"output/cluster_analysis/m2a_clusters_rb.json").read_text())

cluster_id = 511  # change as needed

# clusters is already loaded from playtime_metacritic_clusters
cluster = next(c for c in clusters["clusters"] if c["cluster_id"] == cluster_id)

rows = []
for ent in cluster["entities"]:
    # entity string matches df["id"] exactly, e.g. "metacritic_4525"
    source = ent.split("_", 1)[0]
    df = {"metabooks": metabooks_sample, "amazon": amazon_sample}.get(source)
    hit = df[df["id"] == ent].copy()
    hit.insert(0, "entity", ent)
    hit.insert(1, "source", source)
    rows.append(hit)

entity_rows = pd.concat(rows, ignore_index=True)
display(entity_rows[["entity","clean_title","title","clean_author","publish_year","publisher","isbn_clean"]])
display(pd.DataFrame(cluster["correspondences"]))

,entity,clean_title,title,clean_author,publish_year,publisher,isbn_clean
0,amazon_137383,the fellowship of the ring the lord of the rings part 1,"The Fellowship of the Ring (The Lord of the Rings, Part 1)",j r r tolkien,2003,Houghton Mifflin Company,0618346252
1,amazon_159981,the fellowship of the ring the lord of the rings part 1,"The Fellowship of the Ring (The Lord of the Rings, Part 1)",j r r tolkien,2001,Houghton Mifflin Company,0618153985
2,amazon_16341,the fellowship of the ring lord of the rings paperback,The Fellowship of the Ring (Lord of the Rings (Paperback)),j r r tolkien,1981,Ballantine Books,0345296052
3,amazon_6514,the fellowship of the ring,The Fellowship of the Ring,j r r tolkien,1977,Not Avail,0345272587
4,metabooks_11699,the fellowship of the ring the lord of the rings part 1,"The Fellowship of the Ring (The Lord of the Rings, Part 1)",j r r tolkien,<NA>,William Morrow,0618153985
5,metabooks_196694,the fellowship of the ring the lord of the rings part 1,"The Fellowship of the Ring (The Lord of the Rings, Part 1)",j r r tolkien,<NA>,Houghton Mifflin,0618346252
6,metabooks_350806,fellowship of the ring,Fellowship of the Ring,j r r tolkien,1977,Ballantine Books,0345272587
7,metabooks_469114,fellowship of the ring tv tiein the the lord of the rings 1,"Fellowship of the Ring [TV Tie-In], The (The Lord of the Rings, 1)",j r r tolkien,<NA>,William Morrow Paperbacks,0063270889
8,metabooks_56907,the fellowship of the ring the lord of the rings part 1,"The Fellowship of the Ring (The Lord of the Rings, Part 1)",j r r tolkien,1983,Ballantine Books,0345296052


,entity1,entity2,score,notes
0,amazon_137383,metabooks_11699,0.800000,comparators=4
1,amazon_137383,metabooks_196694,0.881650,comparators=4
2,amazon_137383,metabooks_469114,0.740352,comparators=4
3,amazon_137383,metabooks_56907,0.800000,comparators=4
4,amazon_159981,metabooks_11699,0.800000,comparators=4
5,amazon_159981,metabooks_196694,0.881650,comparators=4
6,amazon_16341,metabooks_350806,0.716228,comparators=4
7,amazon_16341,metabooks_56907,0.810792,comparators=4
8,amazon_6514,metabooks_350806,0.847214,comparators=4


In [45]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a = clusterer.cluster(correspondences_m2a)
mbm_correspondences_m2g = clusterer.cluster(correspondences_m2g)

all_correspondences = pd.concat([mbm_correspondences_m2a, mbm_correspondences_m2g], ignore_index=True)

len(all_correspondences)

28234

In [50]:
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=mbm_correspondences_m2a,
    test_pairs=test_m2a
)

display(eval_results_m2a)
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=mbm_correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 0.9794520547945206,
 'recall': 0.8614457831325302,
 'f1': 0.9166666666666667,
 'accuracy': 0.8930041152263375,
 'true_positives': 143,
 'false_positives': 3,
 'false_negatives': 23,
 'true_negatives': 74,
 'threshold_used': 0.0,
 'total_correspondences': 22553,
 'filtered_correspondences': 22553,
 'evaluation_timestamp': '2025-12-20T03:10:28.821187'}

{'precision': 0.9411764705882353,
 'recall': 0.7777777777777778,
 'f1': 0.8517110266159694,
 'accuracy': 0.8368200836820083,
 'true_positives': 112,
 'false_positives': 7,
 'false_negatives': 32,
 'true_negatives': 88,
 'threshold_used': 0.0,
 'total_correspondences': 5681,
 'filtered_correspondences': 5681,
 'evaluation_timestamp': '2025-12-20T03:10:28.977060',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [46]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust,voting
import pandas as pd

metabooks_sample.attrs["trust_score"] = 3
goodreads_sample.attrs["trust_score"] = 2
amazon_sample.attrs["trust_score"] = 1
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('author', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publish_year', voting)
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('language', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres',union)

In [47]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_rb_emblocker = engine.run(
    datasets=[amazon_sample, metabooks_sample, goodreads_sample],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False
)

print(f'Fused rows: {len(fused_rb_emblocker):,}')


Fused rows: 26,208


In [48]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd


def categories_set_equal(a, b) -> bool:
    """Return True if a and b contain the same unique categories (order/type agnostic)."""
    def to_set(x):
        def items(v):
            # missing
            if v is None or (isinstance(v, float) and np.isnan(v)): return []
            # numpy array → recurse over elements
            if isinstance(v, np.ndarray): 
                out=[]; [out.extend(items(e)) for e in v.flatten()]; return out
            # python containers → recurse over elements
            if isinstance(v, (list, tuple, set)):
                out=[]; [out.extend(items(e)) for e in v]; return out
            # scalar/string: try parse stringified list; else split by delimiters
            s = str(v).strip()
            if s == "" or s.lower() in {"nan","none"}: return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)): return items(parsed)
            except Exception:
                pass
            return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]
        return {it.lower() for it in items(x)}
    return to_set(a) == to_set(b)

fused_dataset = load_parquet(PIPELINE_DIR / "fused_rb_emblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= load_csv(MLDS_DIR / "golden_fused_books.csv")

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("language", tokenized_match)
strategy.add_evaluation_function("genres", categories_set_equal)

In [49]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.921
  macro_accuracy: 0.921
  num_evaluated_records: 20
  num_evaluated_attributes: 7
  total_evaluations: 140
  total_correct: 129
  publish_year_accuracy: 0.950
  publish_year_count: 20
  page_count_accuracy: 1.000
  page_count_count: 20
  author_accuracy: 0.950
  author_count: 20
  language_accuracy: 1.000
  language_count: 20
  title_accuracy: 0.650
  title_count: 20
  publisher_accuracy: 0.950
  publisher_count: 20
  genres_accuracy: 0.950
  genres_count: 20

Overall Accuracy: 92.1%
